# 02C - Asset / Equipment Finance LGD

This notebook adds a dedicated asset/equipment-finance LGD segment under the parent commercial cash-flow framework.

Asset categories covered:
- vehicles
- standard equipment
- specialised machinery


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_commercial_data
from src.commercial_data_controls import assign_framework_segment, run_commercial_data_controls
from src.lgd_calculation import exposure_weighted_average
from src.reproducibility import set_global_seed

set_global_seed(72)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
TABLE_DIR = os.path.join(OUTPUT_DIR, 'tables')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy:** observed workout inputs -> approved proxies -> conservative fallback with disclosure.
- **Proxy logic:** asset type, condition, secondary liquidity, and remarketing assumptions are synthetic portfolio proxies.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status:** portfolio-project baseline only; not calibrated to internal repossession and resale history.


## Objective
Build an interview-ready asset/equipment finance LGD segment where recoveries are tied to repossession and remarketing of financed assets.

## Drivers
- asset type and age
- residual / balloon exposure
- repossession and enforcement cost
- remarketing discount and secondary-market liquidity
- condition/location and borrower operational dependence

## Logic
- EAD reflects instalment-style drawn balances plus modest residual/headroom crystallisation.
- LGD is based on discounted sale proceeds net of repossession/holding costs and legal processing costs.

## Downturn
- Stress assumes weaker resale liquidity, higher remarketing haircuts, and longer repossession-to-sale timing.

## Validation
- Range checks, balance checks, and timing sanity checks are exported for reviewer governance.

## Limitations
- Portfolio-project proxy assumptions are used; production calibration requires internal repossession, resale, and deficiency data.


## Why Asset Finance Differs from Generic Secured Commercial Lending

- Recovery is tied to **specific financed assets** rather than broad enterprise collateral packages.
- Asset age, condition, and secondary-market depth directly affect resale proceeds.
- Residual/balloon exposures can increase loss severity if remarketing outcomes are weak.
- Workout path is typically repossession-and-sale focused, often shorter but more collateral-quality sensitive.


In [ ]:
# Base commercial portfolio and asset/equipment segment selection
all_loans, all_cashflows = generate_commercial_data(n_loans=420, seed=43)
all_loans = all_loans.copy()

all_loans['icr'] = pd.to_numeric(all_loans['interest_coverage_ratio'], errors='coerce')
all_loans['industry_risk_bucket'] = all_loans['industry_risk_level'].fillna('Medium')
all_loans['watchlist_flag'] = (
    all_loans['default_trigger'].isin(['Covenant Breach', 'Voluntary Administration', 'Receivership'])
    | (all_loans['icr'] < 1.35)
).astype(int)
all_loans['framework_segment'] = assign_framework_segment(all_loans)

asset = all_loans[all_loans['framework_segment'] == 'Asset / Equipment Finance'].copy()

if len(asset) == 0:
    raise ValueError('No asset/equipment segment rows found for current synthetic sample.')

print('All loans:', len(all_loans))
print('Asset/equipment segment loans:', len(asset))
display(asset[['loan_id', 'facility_type', 'security_type', 'ead', 'drawn_balance', 'undrawn_amount']].head(10))


In [ ]:
# Asset drivers and EAD logic
rng = np.random.default_rng(72)

asset['asset_type'] = rng.choice(
    ['Vehicles', 'Standard Equipment', 'Specialised Machinery'],
    size=len(asset),
    p=[0.45, 0.35, 0.20],
)

type_age = {
    'Vehicles': (1.0, 7.0),
    'Standard Equipment': (2.0, 10.0),
    'Specialised Machinery': (3.0, 14.0),
}
asset['asset_age_years'] = [rng.uniform(*type_age[t]) for t in asset['asset_type']]

asset['residual_balloon_pct'] = (
    0.08
    + 0.10 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.05 * asset['asset_type'].eq('Vehicles').astype(int)
    + rng.normal(0.0, 0.03, len(asset))
).clip(0.00, 0.35)

asset['repossession_cost_pct'] = (
    0.020
    + 0.018 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.012 * asset['watchlist_flag']
    + rng.normal(0.0, 0.005, len(asset))
).clip(0.010, 0.090)

asset['remarketing_discount_pct'] = (
    0.120
    + 0.090 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.050 * (asset['asset_age_years'] / 12.0).clip(0.0, 1.0)
    + 0.040 * asset['watchlist_flag']
    + rng.normal(0.0, 0.020, len(asset))
).clip(0.05, 0.50)

asset['secondary_market_liquidity'] = (
    0.78
    - 0.32 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    - 0.10 * (asset['asset_age_years'] / 12.0).clip(0.0, 1.0)
    + rng.normal(0.0, 0.06, len(asset))
).clip(0.10, 0.98)

asset['asset_condition_location_score'] = (
    0.70
    - 0.10 * (asset['asset_age_years'] / 12.0).clip(0.0, 1.0)
    - 0.08 * asset['watchlist_flag']
    + rng.normal(0.0, 0.06, len(asset))
).clip(0.10, 0.98)

asset['operational_dependence_score'] = (
    0.45
    + 0.22 * asset['asset_type'].eq('Specialised Machinery').astype(int)
    + 0.08 * asset['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)
    + rng.normal(0.0, 0.06, len(asset))
).clip(0.05, 0.98)

asset['liquidity_band'] = pd.cut(
    asset['secondary_market_liquidity'],
    bins=[0.0, 0.35, 0.60, 0.80, 1.00],
    labels=['Very Low', 'Low', 'Medium', 'High'],
    include_lowest=True,
)

asset['ead_base'] = (
    asset['drawn_balance']
    + 0.20 * asset['ccf'] * asset['undrawn_amount']
    + 0.18 * asset['residual_balloon_pct'] * asset['facility_limit']
)
asset['ead_base'] = np.maximum(asset['ead_base'], asset['drawn_balance'])
asset['ead_base'] = np.minimum(asset['ead_base'], asset['facility_limit'] * 1.05)

asset['ead_downturn'] = (
    asset['ead_base']
    * (1.02 + 0.04 * asset['watchlist_flag'] + 0.03 * asset['operational_dependence_score'])
)
asset['ead_downturn'] = np.maximum(asset['ead_downturn'], asset['drawn_balance'])
asset['ead_downturn'] = np.minimum(asset['ead_downturn'], asset['facility_limit'] * 1.10)

asset['ead_uplift_pct'] = ((asset['ead_downturn'] - asset['ead_base']) / asset['ead_base'].replace(0, np.nan)).fillna(0.0).clip(0.0, 1.5)

ead_summary = (
    asset.groupby('asset_type', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_drawn': g['drawn_balance'].sum(),
                'total_ead_base': g['ead_base'].sum(),
                'total_ead_downturn': g['ead_downturn'].sum(),
                'ead_weighted_uplift_pct': exposure_weighted_average(g, 'ead_uplift_pct', 'ead_base'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('total_ead_base', ascending=False)
)

print('=== Asset/Eq EAD Summary ===')
display(ead_summary.round(4))


In [ ]:
# Recovery and LGD logic (base/downturn/final)
specialised_flag = asset['asset_type'].eq('Specialised Machinery').astype(int)
vehicle_flag = asset['asset_type'].eq('Vehicles').astype(int)

asset_age_scaled = (asset['asset_age_years'] / 12.0).clip(0.0, 1.2)

asset['repossession_months'] = (
    2.5
    + 4.0 * (1.0 - asset['secondary_market_liquidity'])
    + 2.0 * specialised_flag
    + 1.5 * asset['watchlist_flag']
    + 1.2 * asset_age_scaled
    + np.random.default_rng(721).normal(0.0, 0.8, len(asset))
).clip(1.0, 24.0)

asset['sale_timing_months'] = (
    asset['repossession_months']
    + 1.5
    + 3.0 * (1.0 - asset['secondary_market_liquidity'])
    + 1.5 * specialised_flag
).clip(2.0, 36.0)

asset['asset_haircut_proxy'] = (
    asset['remarketing_discount_pct']
    + 0.10 * specialised_flag
    + 0.06 * asset_age_scaled
    + 0.05 * (1.0 - asset['asset_condition_location_score'])
).clip(0.08, 0.70)

asset['sale_proceeds_pct_of_ead'] = (
    (1.0 - asset['asset_haircut_proxy'])
    * (0.62 + 0.30 * asset['secondary_market_liquidity'])
    * (0.65 + 0.30 * asset['asset_condition_location_score'])
).clip(0.05, 0.98)

asset['discount_factor_proxy'] = (1.0 + asset['discount_rate']) ** (asset['sale_timing_months'] / 12.0)
asset['discounted_recovery_pct'] = (asset['sale_proceeds_pct_of_ead'] / asset['discount_factor_proxy']).clip(0.03, 0.98)

asset['enforcement_holding_cost_pct'] = (
    asset['repossession_cost_pct']
    + 0.012
    + 0.010 * (asset['sale_timing_months'] / 24.0).clip(0.0, 1.5)
    + 0.012 * specialised_flag
).clip(0.015, 0.18)

asset['liquidation_loss_proxy'] = (
    1.0
    - asset['discounted_recovery_pct']
    + asset['enforcement_holding_cost_pct']
    + 0.05 * asset['residual_balloon_pct']
    + 0.04 * asset['operational_dependence_score']
).clip(0.05, 0.99)

asset['lgd_base_economic'] = (
    0.45 * asset['realised_lgd']
    + 0.55 * asset['liquidation_loss_proxy']
).clip(0.05, 0.95)

asset['sale_timing_months_downturn'] = (
    asset['sale_timing_months']
    * (1.10 + 0.08 * specialised_flag + 0.04 * (1.0 - asset['secondary_market_liquidity']))
).clip(2.0, 48.0)

asset['downturn_scalar'] = (
    1.00
    + 0.16 * asset['asset_haircut_proxy']
    + 0.08 * ((asset['sale_timing_months_downturn'] - asset['sale_timing_months']) / 24.0).clip(0.0, 1.0)
    + 0.06 * asset['operational_dependence_score']
    + 0.03 * vehicle_flag
).clip(1.00, 1.55)

asset['lgd_downturn'] = (asset['lgd_base_economic'] * asset['downturn_scalar']).clip(0.0, 1.0)

asset['moc_addon'] = 0.020 + 0.010 * asset['watchlist_flag']
asset['supervisory_floor_proxy'] = asset['asset_type'].map({
    'Vehicles': 0.25,
    'Standard Equipment': 0.35,
    'Specialised Machinery': 0.45,
}).fillna(0.35)
asset['lgd_final_regulatory'] = np.maximum(asset['lgd_downturn'] + asset['moc_addon'], asset['supervisory_floor_proxy']).clip(0.0, 1.0)

def weighted_lgd(df, group_col):
    rows = []
    for k, g in df.groupby(group_col, observed=True):
        rows.append({
            'dimension': group_col,
            'bucket': str(k),
            'loan_count': len(g),
            'total_ead_base': g['ead_base'].sum(),
            'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_economic', 'ead_base'),
            'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead_base'),
            'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
        })
    return pd.DataFrame(rows)

weighted_by_asset = weighted_lgd(asset, 'asset_type')
weighted_by_liquidity = weighted_lgd(asset, 'liquidity_band')
segment_summary = pd.concat([weighted_by_asset, weighted_by_liquidity], ignore_index=True)

base_vs_downturn = pd.DataFrame([
    {
        'ead_weighted_lgd_base': exposure_weighted_average(asset, 'lgd_base_economic', 'ead_base'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(asset, 'lgd_downturn', 'ead_base'),
        'ead_weighted_lgd_final': exposure_weighted_average(asset, 'lgd_final_regulatory', 'ead_base'),
    }
])
base_vs_downturn['downturn_sensitivity_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_downturn'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)
base_vs_downturn['final_minus_base_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_final'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)

print('=== Asset/Eq Weighted LGD by Asset Type ===')
display(weighted_by_asset.round(4))
print('=== Asset/Eq Weighted LGD by Liquidity Band ===')
display(weighted_by_liquidity.round(4))
print('=== Base vs Downturn ===')
display(base_vs_downturn.round(4))


In [ ]:
# Sensitivity analysis: remarketing discount and liquidity
def run_asset_sensitivity(df, discount_add=0.0, liquidity_shift=0.0):
    x = df.copy()
    remarketing = (x['remarketing_discount_pct'] + discount_add).clip(0.01, 0.80)
    liquidity = (x['secondary_market_liquidity'] + liquidity_shift).clip(0.05, 1.00)

    haircut = (
        remarketing
        + 0.10 * x['asset_type'].eq('Specialised Machinery').astype(int)
        + 0.06 * (x['asset_age_years'] / 12.0).clip(0.0, 1.2)
        + 0.05 * (1.0 - x['asset_condition_location_score'])
    ).clip(0.08, 0.80)

    proceeds = (
        (1.0 - haircut)
        * (0.62 + 0.30 * liquidity)
        * (0.65 + 0.30 * x['asset_condition_location_score'])
    ).clip(0.03, 0.98)

    rec_months = (x['sale_timing_months'] * (1.00 + 0.25 * discount_add - 0.20 * liquidity_shift)).clip(2.0, 52.0)
    disc = (1.0 + x['discount_rate']) ** (rec_months / 12.0)

    lgd_base_s = (
        0.45 * x['realised_lgd']
        + 0.55 * (
            1.0
            - (proceeds / disc)
            + x['enforcement_holding_cost_pct']
            + 0.05 * x['residual_balloon_pct']
            + 0.04 * x['operational_dependence_score']
        )
    ).clip(0.05, 0.97)

    lgd_down_s = (
        lgd_base_s
        * (1.00 + 0.16 * haircut + 0.05 * x['operational_dependence_score'])
    ).clip(0.0, 1.0)

    lgd_final_s = np.maximum(
        lgd_down_s + 0.020 + 0.010 * x['watchlist_flag'],
        x['supervisory_floor_proxy'],
    ).clip(0.0, 1.0)

    y = x.assign(lgd_base_s=lgd_base_s, lgd_down_s=lgd_down_s, lgd_final_s=lgd_final_s)
    return {
        'ead_weighted_lgd_base': exposure_weighted_average(y, 'lgd_base_s', 'ead_base'),
        'ead_weighted_lgd_downturn': exposure_weighted_average(y, 'lgd_down_s', 'ead_base'),
        'ead_weighted_lgd_final': exposure_weighted_average(y, 'lgd_final_s', 'ead_base'),
    }

scenarios = [
    {'scenario': 'base', 'discount_add': 0.00, 'liquidity_shift': 0.00},
    {'scenario': 'remarketing_discount_+10pp', 'discount_add': 0.10, 'liquidity_shift': 0.00},
    {'scenario': 'liquidity_-10pp', 'discount_add': 0.00, 'liquidity_shift': -0.10},
    {'scenario': 'combined_stress', 'discount_add': 0.10, 'liquidity_shift': -0.10},
]

sensitivity_rows = []
for s in scenarios:
    m = run_asset_sensitivity(asset, discount_add=s['discount_add'], liquidity_shift=s['liquidity_shift'])
    sensitivity_rows.append({'scenario': s['scenario'], **m})

sensitivity = pd.DataFrame(sensitivity_rows)
print('=== Asset/Eq Sensitivity ===')
display(sensitivity.round(4))


In [ ]:
# Monitoring, validation, charts, exports
asset['default_year'] = pd.to_datetime(asset['default_date']).dt.year
monitoring = (
    asset.groupby('default_year', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead_base': g['ead_base'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead_base'),
                'mean_sale_timing_months': g['sale_timing_months'].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

core_controls = run_commercial_data_controls(
    asset,
    None,
    segment_col='framework_segment',
    extra_probability_cols=['asset_condition_location_score', 'operational_dependence_score'],
    extra_haircut_cols=['remarketing_discount_pct', 'asset_haircut_proxy', 'residual_balloon_pct'],
)

extra_checks = pd.DataFrame([
    {'check_name': 'ead_base_not_below_drawn', 'passed': (asset['ead_base'] >= asset['drawn_balance']).all(), 'failed_rows': int((asset['ead_base'] < asset['drawn_balance']).sum()), 'detail': 'ead_base >= drawn_balance'},
    {'check_name': 'ead_downturn_not_below_drawn', 'passed': (asset['ead_downturn'] >= asset['drawn_balance']).all(), 'failed_rows': int((asset['ead_downturn'] < asset['drawn_balance']).sum()), 'detail': 'ead_downturn >= drawn_balance'},
    {'check_name': 'downturn_sale_timing_not_shorter', 'passed': (asset['sale_timing_months_downturn'] >= asset['sale_timing_months']).all(), 'failed_rows': int((asset['sale_timing_months_downturn'] < asset['sale_timing_months']).sum()), 'detail': 'downturn sale timing >= base sale timing'},
    {'check_name': 'lgd_downturn_not_below_base', 'passed': (asset['lgd_downturn'] >= asset['lgd_base_economic']).all(), 'failed_rows': int((asset['lgd_downturn'] < asset['lgd_base_economic']).sum()), 'detail': 'downturn LGD >= base LGD'},
])

validation_checks = pd.concat([core_controls, extra_checks], ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(weighted_by_asset['bucket'], weighted_by_asset['ead_weighted_lgd_final'], color='#4c72b0')
ax.set_title('Asset/Equipment LGD: Weighted Final LGD by Asset Type')
ax.set_xlabel('Asset Type')
ax.set_ylabel('EAD-weighted Final LGD')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'asset_equipment_weighted_lgd_by_asset_type.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(ead_summary['asset_type'], ead_summary['ead_weighted_uplift_pct'], color='#55a868')
ax.set_title('Asset/Equipment EAD Uplift by Asset Type')
ax.set_xlabel('Asset Type')
ax.set_ylabel('EAD Uplift %')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'asset_equipment_ead_uplift_by_asset_type.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

loan_level_output = asset[[
    'loan_id', 'facility_type', 'security_type', 'framework_segment', 'industry', 'industry_risk_bucket',
    'asset_type', 'asset_age_years', 'residual_balloon_pct', 'repossession_cost_pct',
    'remarketing_discount_pct', 'secondary_market_liquidity', 'asset_condition_location_score',
    'operational_dependence_score', 'liquidity_band', 'drawn_balance', 'undrawn_amount',
    'ead_base', 'ead_downturn', 'ead_uplift_pct', 'repossession_months', 'sale_timing_months',
    'sale_timing_months_downturn', 'asset_haircut_proxy', 'discounted_recovery_pct',
    'enforcement_holding_cost_pct', 'realised_lgd', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory',
]].copy()

loan_level_output.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_segment_summary.csv'), index=False)
ead_summary.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_ead_summary.csv'), index=False)
base_vs_downturn.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_base_vs_downturn.csv'), index=False)
sensitivity.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_sensitivity.csv'), index=False)
monitoring.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_monitoring_by_year.csv'), index=False)
validation_checks.to_csv(os.path.join(TABLE_DIR, 'asset_equipment_finance_validation_checks.csv'), index=False)

print('=== Validation Checks ===')
display(validation_checks)

print('Saved asset/equipment outputs:')
print('- asset_equipment_finance_loan_level_output.csv')
print('- asset_equipment_finance_segment_summary.csv')
print('- asset_equipment_finance_ead_summary.csv')
print('- asset_equipment_finance_base_vs_downturn.csv')
print('- asset_equipment_finance_sensitivity.csv')
print('- asset_equipment_finance_monitoring_by_year.csv')
print('- asset_equipment_finance_validation_checks.csv')


---

## APS 113 Calibration Layer — Asset & Equipment Finance

> **SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY**
>
> This section adds a full APS 113-aligned calibration loop on top of the
> proxy baseline above. Workout data is synthetically generated (2014-2024,
> 10-year window) to demonstrate methodology; real production calibration
> requires an internal workout tape.

### Calibration Pipeline (APS 113 s.32-68)

| Step | Function | APS 113 |
|------|----------|---------|
| 1 | Load/generate historical workout data | s.44, Att A |
| 2 | `compute_realised_lgd()` — LIP costs, cure leg | s.32-34 |
| 3 | `classify_economic_regime()` + `assign_regime_to_workouts()` | s.43, s.46 |
| 4 | `segment_lgd()` — product-specific segment keys | s.52 |
| 5 | `compute_long_run_lgd()` — vintage-EWA method | s.43-44 |
| 6 | `compare_model_vs_actual()` — proxy vs realised | s.60-62 |
| 7 | `apply_downturn_overlay()` + `apply_correlation_adjustment()` | s.46-50, s.55-57 |
| 8 | `MoCRegister` + `apply_moc()` — 5 APS 113 s.65 sources | s.63-65 |
| 9 | `apply_regulatory_floor()` — 20% per APS 113 s.58 — depreciating collateral | s.58 |
| 10 | Export 9 CSV outputs | — |
| 11 | `run_full_validation_suite()` — Gini, HL, PSI, OOT | s.66-68 |

**Correct APS 113 order:** LR-LGD → downturn overlay → correlation adj →
MoC → floor (MoC applied to downturn LGD, not long-run LGD, per s.63).


In [ ]:
# APS 113 Calibration Layer — imports and configuration
import os, sys
from pathlib import Path
sys.path.insert(0, os.path.abspath('..'))

from src.calibration_utils import (
    compute_realised_lgd,
    segment_lgd,
    compute_long_run_lgd,
    compare_model_vs_actual,
    classify_economic_regime,
    assign_regime_to_workouts,
    apply_downturn_overlay,
    apply_correlation_adjustment,
    build_lgd_pd_annual_series,
    estimate_lgd_pd_correlation,
    MoCRegister,
    apply_moc,
    apply_regulatory_floor,
    run_full_validation_suite,
    generate_compliance_map,
    export_compliance_map,
    CALIBRATION_STEP_ORDER,
)
from src.generators import GENERATOR_MAP, generate_all_historical_workouts

PRODUCT = "asset_equipment"
SEGMENT_KEYS = ['asset_class', 'secondary_market_liquidity']
MODEL_LGD_COL = "lgd_final"

HISTORY_DIR = Path('..') / 'data' / 'generated' / 'historical'
OUTPUTS_DIR = Path('..') / 'outputs' / 'tables'
UPSTREAM_PARQUET = Path('..') / 'data' / 'exports' / 'macro_regime_flags.parquet'

HISTORY_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Product: {PRODUCT}")
print(f"Segment keys: {SEGMENT_KEYS}")
print(f"APS 113 calibration pipeline — step order:")
for step, fn, ref in CALIBRATION_STEP_ORDER:
    print(f"  Step {step:>2}: {fn:<35} {ref}")


In [ ]:
# ── STEP 1: Load or generate historical workout data (APS 113 s.44, Att A) ─
parquet_path = HISTORY_DIR / f"{PRODUCT}_workouts.parquet"

if parquet_path.exists():
    cal_loans = pd.read_parquet(parquet_path)
    # cashflows stored alongside or re-generated
    try:
        cal_cashflows = pd.read_parquet(
            HISTORY_DIR / f"{PRODUCT}_cashflows.parquet"
        )
    except FileNotFoundError:
        cal_cashflows = None
    print(f"Loaded {len(cal_loans):,} historical workout loans from Parquet")
else:
    print(f"Parquet not found — generating synthetic workout data for {PRODUCT}")
    results = generate_all_historical_workouts(
        seed=42, output_dir=HISTORY_DIR, write_parquet=True, products=[PRODUCT]
    )
    cal_loans = results[PRODUCT]["loans"]
    cal_cashflows = results[PRODUCT].get("cashflows")
    print(f"Generated {len(cal_loans):,} synthetic workout loans (2014-2024)")

print(f"Date range: {cal_loans['default_date'].min()} to {cal_loans['default_date'].max()}")
print(f"Columns: {list(cal_loans.columns)}")

# ── STEP 2: Compute realised LGD (APS 113 s.32-34) ─────────────────────────
# LIP costs (Loss Identification Period, ~90 days) auto-detected if cashflows available
lgd_df = compute_realised_lgd(
    loans=cal_loans,
    cashflows=cal_cashflows,
    ead_col="ead_at_default",
    recovery_col="recovery_amount",
    cost_col="direct_costs",
    lip_window_days=90,
)
print(f"\nStep 2: Realised LGD computed")
print(f"  EAD-weighted realised LGD: {(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum():.2%}")
display(lgd_df[['realised_lgd', 'ead_at_default']].describe().round(4))

# ── STEP 3: Economic regime classification (APS 113 s.43, s.46-50) ─────────
upstream_path = str(UPSTREAM_PARQUET) if UPSTREAM_PARQUET.exists() else None
regime_df = classify_economic_regime(
    upstream_parquet_path=upstream_path,
    method="upstream_first",
)
print(f"\nStep 3: Economic regimes classified")
print(f"  Data source: {regime_df['data_source'].iloc[0]}")
display(regime_df[['year', 'regime', 'is_downturn_period', 'data_source']].head(12))

lgd_df = assign_regime_to_workouts(lgd_df, regime_df)
downturn_pct = lgd_df.get('is_downturn_period', pd.Series([False])).mean()
print(f"  Downturn observations: {downturn_pct:.1%} of portfolio")

# ── STEP 4: Segment by product-specific keys (APS 113 s.52) ────────────────
segmented_df = segment_lgd(lgd_df, SEGMENT_KEYS)
low_count = segmented_df[segmented_df.get('segment_flag', '') == 'low_count'] if 'segment_flag' in segmented_df.columns else pd.DataFrame()
print(f"\nStep 4: Segmentation complete")
print(f"  Segments: {segmented_df.groupby(SEGMENT_KEYS, observed=True).ngroups}")
if not low_count.empty:
    print(f"  Low-count segments flagged (<20 obs): {len(low_count)}")

# ── STEP 5: Long-run LGD — vintage-EWA (APS 113 s.43-44) ─────────────────
lr_lgd_df = compute_long_run_lgd(
    segmented_df,
    segment_keys=SEGMENT_KEYS,
    method="vintage_ewa",
)
print(f"\nStep 5: Long-run LGD by segment (vintage-EWA)")
display(lr_lgd_df.round(4))
lr_lgd_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_long_run_lgd_by_segment.csv", index=False)

# ── STEP 6: Compare model vs actual (APS 113 s.60-62) ──────────────────────
# 'model_lgd' here = proxy model LGD from the section above (lgd_final if present)
if MODEL_LGD_COL in cal_loans.columns:
    compare_input = lgd_df.merge(
        cal_loans[['loan_id', MODEL_LGD_COL] if 'loan_id' in cal_loans.columns else [MODEL_LGD_COL]],
        left_index=True, right_index=True, how='left',
    ) if MODEL_LGD_COL not in lgd_df.columns else lgd_df
else:
    compare_input = lgd_df.copy()
    compare_input['model_lgd'] = lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else 0.25

model_col_to_use = MODEL_LGD_COL if MODEL_LGD_COL in compare_input.columns else 'model_lgd'
mva_df = compare_model_vs_actual(
    compare_input,
    model_lgd_col=model_col_to_use,
    segment_keys=SEGMENT_KEYS,
)
print(f"\nStep 6: Model vs actual comparison")
display(mva_df.round(4))
mva_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_model_vs_actual_comparison.csv", index=False)

# ── STEP 7: Downturn overlay + Frye-Jacobs correlation adj (s.46-50, s.55-57)
# Reuses apply_downturn_overlay from existing lgd_calculation.py
dt_lgd = apply_downturn_overlay(segmented_df, product=PRODUCT)
print(f"\nStep 7: Downturn overlay applied")
if 'lgd_downturn' in dt_lgd.columns:
    ewa_dt = (dt_lgd['lgd_downturn'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    ewa_lr = (dt_lgd['realised_lgd'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
    print(f"  EWA Long-run LGD:  {ewa_lr:.2%}")
    print(f"  EWA Downturn LGD:  {ewa_dt:.2%}")
    downturn_col = 'lgd_downturn'
else:
    dt_lgd['lgd_downturn'] = dt_lgd['realised_lgd'] * 1.15  # fallback scalar
    downturn_col = 'lgd_downturn'

# Frye-Jacobs correlation adjustment (APS 113 s.55-57)
try:
    lgd_ts, pd_ts = build_lgd_pd_annual_series(dt_lgd)
    macro_for_corr = regime_df.rename(columns={'gdp_growth': 'gdp_growth_yoy'})
    corr_result = estimate_lgd_pd_correlation(lgd_ts, pd_ts, macro_for_corr)
    dt_lgd['lgd_downturn_corr_adj'] = apply_correlation_adjustment(
        dt_lgd[downturn_col], corr_result['rho'], corr_result['macro_shock_std']
    )
    downturn_col = 'lgd_downturn_corr_adj'
    print(f"  Frye-Jacobs rho={corr_result['rho']:.3f}, adj factor={corr_result['lgd_dt_adjustment_factor']:.3f}")
except Exception as exc:
    print(f"  Frye-Jacobs skipped: {exc}")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_downturn_lgd_by_segment.csv", index=False)

# ── STEP 8: MoC register + apply MoC (AFTER downturn — APS 113 s.63-65) ───
# Determine regime data source for MoC model_approximation component
data_src = regime_df['data_source'].iloc[0] if 'data_source' in regime_df.columns else 'synthetic'
n_downturn_yrs = int(regime_df['is_downturn_period'].sum()) if 'is_downturn_period' in regime_df.columns else 0

psi_approx = 0.05  # placeholder — full PSI computed in Step 11
bias_approx = float(mva_df['bias'].abs().mean()) if 'bias' in mva_df.columns else 0.02

moc_register = MoCRegister(product=PRODUCT, regime_data_source=data_src)
moc_df = moc_register.build_moc_register(
    segment_df=segmented_df,
    segment_keys=SEGMENT_KEYS,
    n_downturn_vintages=n_downturn_yrs,
    psi_value=psi_approx,
    backtesting_bias=bias_approx,
)
print(f"\nStep 8: MoC register built")
display(moc_df.round(4))
moc_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_moc_register.csv", index=False)

lgd_with_moc = apply_moc(dt_lgd[downturn_col], moc_df, segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None)
dt_lgd['lgd_with_moc'] = lgd_with_moc
moc_ewa = (lgd_with_moc * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
print(f"  EWA LGD after MoC: {moc_ewa:.2%}")

# ── STEP 9: Regulatory floors (APS 113 s.58) ──────────────────────────────
dt_lgd['lgd_final_calibrated'] = apply_regulatory_floor(dt_lgd['lgd_with_moc'], product=PRODUCT)
final_ewa = (dt_lgd['lgd_final_calibrated'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()
floor_binding_pct = (dt_lgd['lgd_final_calibrated'] > dt_lgd['lgd_with_moc']).mean()
print(f"\nStep 9: Regulatory floor applied")
print(f"  EWA Final Calibrated LGD: {final_ewa:.2%}")
print(f"  Floor binding for: {floor_binding_pct:.1%} of loans")

dt_lgd.to_csv(OUTPUTS_DIR / f"{PRODUCT}_final_calibrated_lgd.csv", index=False)

# ── STEP 10: Export all outputs ────────────────────────────────────────────
# Already exported: long_run_lgd_by_segment, model_vs_actual, downturn_lgd, moc_register, final_calibrated_lgd
# Export remaining:
lgd_df[['realised_lgd', 'ead_at_default']].assign(product=PRODUCT).to_csv(
    OUTPUTS_DIR / f"{PRODUCT}_historical_workouts.csv", index=False
)
regime_df.to_csv(OUTPUTS_DIR / f"{PRODUCT}_regime_classification.csv", index=False)

# Calibration adjustments summary
cal_adj_summary = pd.DataFrame({
    'product': [PRODUCT],
    'ewa_realised_lgd': [(lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum()],
    'ewa_long_run_lgd': [lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None],
    'ewa_downturn_lgd': [(dt_lgd.get('lgd_downturn', dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_with_moc': [(dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum()],
    'ewa_lgd_final': [final_ewa],
    'floor_binding_pct': [floor_binding_pct],
    'regime_data_source': [data_src],
    'n_loans': [len(lgd_df)],
    'calibration_date': [pd.Timestamp.today().date()],
})
cal_adj_summary.to_csv(OUTPUTS_DIR / f"{PRODUCT}_calibration_adjustments.csv", index=False)
print(f"\nStep 10: All outputs exported to {OUTPUTS_DIR}")

# ── STEP 11: Full validation suite (APS 113 s.66-68) ──────────────────────
print(f"\nStep 11: Running full validation suite...")
try:
    val_results = run_full_validation_suite(
        loans=dt_lgd,
        predicted_col='lgd_final_calibrated',
        actual_col='realised_lgd',
        segment_col=SEGMENT_KEYS[0] if SEGMENT_KEYS else None,
        date_col='default_date' if 'default_date' in dt_lgd.columns else None,
        product=PRODUCT,
    )
    print(f"  Gini: {val_results.get('gini', 'n/a')}")
    print(f"  Calibration ratio: {val_results.get('calibration_ratio', 'n/a')}")
    print(f"  PSI: {val_results.get('psi', 'n/a')}")
    if 'summary_table' in val_results:
        val_results['summary_table'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_validation_report.csv", index=False
        )
    if 'backtest_results' in val_results:
        val_results['backtest_results'].to_csv(
            OUTPUTS_DIR / f"{PRODUCT}_backtest_results.csv", index=False
        )
except Exception as exc:
    print(f"  Validation suite error (non-fatal): {exc}")


In [ ]:
# ── Calibration summary waterfall ──────────────────────────────────────────
import matplotlib.pyplot as plt

try:
    stages = {
        'Realised LGD\n(2014-2024)': (lgd_df['realised_lgd'] * lgd_df['ead_at_default']).sum() / lgd_df['ead_at_default'].sum(),
        'Long-run LGD\n(vintage-EWA)': lr_lgd_df['long_run_lgd'].mean() if 'long_run_lgd' in lr_lgd_df.columns else None,
        'Downturn LGD': (dt_lgd.get(downturn_col, dt_lgd['realised_lgd']) * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        '+ MoC': (dt_lgd['lgd_with_moc'] * dt_lgd['ead_at_default']).sum() / dt_lgd['ead_at_default'].sum(),
        'Final\n(Floor Applied)': final_ewa,
    }
    labels = [k for k, v in stages.items() if v is not None]
    values = [v for v in stages.values() if v is not None]
    colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c', '#8e44ad'][:len(labels)]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.6)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    ax.set_ylabel('EAD-Weighted LGD')
    ax.set_title(f'APS 113 Calibration Waterfall — Asset & Equipment Finance')
    ax.set_ylim(0, max(values) * 1.35)
    ax.axhline(values[-1], color='black', ls=':', lw=1, label=f'Final: {values[-1]:.1%}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        Path('..') / 'outputs' / 'figures' / f'asset_equipment_calibration_waterfall.png',
        dpi=150, bbox_inches='tight',
    )
    plt.show()
except Exception as exc:
    print(f"Waterfall chart error (non-fatal): {exc}")

# ── APS 113 compliance snapshot ─────────────────────────────────────────────
compliance_df = generate_compliance_map(
    calibration_results={PRODUCT: {'long_run_lgd_by_segment': True, 'calibration_steps': True}},
    moc_registers={PRODUCT: moc_df},
    regime_data_source=data_src,
    products=[PRODUCT],
)
print("\n=== APS 113 Compliance Snapshot ===")
display(compliance_df[['section_ref', 'requirement', 'status', 'reviewer_note']].set_index('section_ref'))
export_compliance_map(compliance_df, OUTPUTS_DIR / f"asset_equipment_aps113_compliance.csv")

# Final summary
print("\n=== Calibration Summary ===")
display(cal_adj_summary.round(4))
print(f"\nAll calibration outputs in: {OUTPUTS_DIR}")
print(f"SYNTHETIC HISTORICAL CALIBRATION DATA — FOR DEMONSTRATION ONLY")
